<a href="https://colab.research.google.com/github/Khoawawa/DeepLearning-HCMUT-ASS-MS/blob/main/ImageClassification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")

In [ ]:
ic_path = "/content/drive/MyDrive/DeepLearning/Assignment1/ImageClassification"
os.makedirs(ic_path, exist_ok=True)

In [12]:
batch_size = 64

In [13]:
import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset

IMAGENET_train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

IMAGENET_test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

train_dataset_full = torchvision.datasets.CIFAR10(
    root="/content/data",
    train=True,
    download=True,
    transform=IMAGENET_train_transform
)

val_dataset_full = torchvision.datasets.CIFAR10(
    root="/content/data",
    train=True,
    download=False,
    transform=IMAGENET_test_transform
)

train_size = int(0.9 * len(train_dataset_full))
val_size = len(train_dataset_full) - train_size

generator = torch.Generator().manual_seed(42)

indices = torch.randperm(len(train_dataset_full), generator=generator)

train_indices = indices[:train_size]
val_indices = indices[train_size:]

trainset = Subset(train_dataset_full, train_indices)
valset = Subset(val_dataset_full, val_indices)

trainloader = DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=2)
valloader = DataLoader(valset, batch_size=batch_size, shuffle=False, num_workers=2)

# test dataset
testset = torchvision.datasets.CIFAR10(
    root="/content/data",
    train=False,
    download=True,
    transform=IMAGENET_test_transform
)

testloader = DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=2)

classes = (
    "airplane","automobile","bird","cat",
    "deer","dog","frog","horse","ship","truck"
)

In [1]:
import time
import numpy as np
import copy

In [2]:
def train(model,train_loader, val_loader, optimizer, criterion,device,num_epoch, patience=3):
  model.to(device)

  history = {
      "train_loss": [],
      "train_acc": [],
      "val_loss": [],
      "val_acc": [],
      "epoch_time": [],
      "avg_epoch_time": None
  }

  best_val_acc = float("-inf")
  patience_c = 0
  best_state = None

  for epoch in range(num_epoch):
    start_time = time.perf_counter()
    model.train()
    # train loop

    train_loss = 0
    correct = 0
    total = 0
    for batch, label in tqdm(train_loader):
      batch = batch.to(device)
      label = label.to(device)

      logits = model(batch)
      loss = criterion(logits, label)

      optimizer.zero_grad()
      loss.backward()
      optimizer.step()

      train_loss += loss.item()

      pred = torch.argmax(logits, dim=1)
      correct += (pred == label).sum().item()
      total += label.size(0)

    train_acc = correct / total
    train_loss /= len(train_loader)

    # val loop
    model.eval()

    val_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
      for batch, label in tqdm(val_loader):

        batch = batch.to(device)
        label = label.to(device)

        logits = model(batch)
        loss = criterion(logits, label)

        val_loss += loss.item()

        pred = torch.argmax(logits, dim=1)
        correct += (pred == label).sum().item()
        total += label.size(0)

    val_acc = correct / total
    val_loss /= len(val_loader)

    epoch_time = time.perf_counter() - start_time
    history["epoch_time"].append(epoch_time)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(f"Epoch {epoch+1}: val_acc = {val_acc:.4f}")

    # Early stopping
    if val_acc > best_val_acc:
      best_val_acc = val_acc
      patience_c = 0
      best_state = copy.deepcopy(model.state_dict())
    else:
      patience_c += 1

      if patience_c >= patience:
        print(f"Early stopping at epoch {epoch+1}")
        model.load_state_dict(best_state)
        break
  history["avg_epoch_time"] = sum(history["epoch_time"]) / len(history["epoch_time"])

  return model, history


In [3]:
import torch
from tqdm import tqdm
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

def eval(model, test_loader,device):

  model.to(device)
  model.eval()
  correct = 0
  total = 0

  all_preds = []
  all_labels = []

  with torch.no_grad():
    for batch, label in tqdm(test_loader):
      batch = batch.to(device)

      logits = model(batch)
      pred = torch.argmax(logits, dim=1).cpu()

      all_preds.extend(pred.numpy())
      all_labels.extend(label.numpy())

  cm = confusion_matrix(all_labels, all_preds)
  acc = accuracy_score(all_labels, all_preds)
  precision = precision_score(all_labels, all_preds, average="macro")
  recall = recall_score(all_labels, all_preds, average="macro")
  f1 = f1_score(all_labels, all_preds, average="macro")

  return {
      "cm": cm,
      "acc": acc,
      "precision": precision,
      "recall": recall,
      "f1": f1
  }

In [4]:
import os
import torch
import numpy as np

def save_result_to_drive(path, model_name, model, history, eval_result):
    model_path = os.path.join(path, model_name)
    os.makedirs(model_path, exist_ok=True)

    # Save model
    torch.save({
        "model_state_dict": model.state_dict(),
        "model_config": str(model),
    }, os.path.join(model_path, "best_model.pth"))

    # Save history
    with open(os.path.join(model_path, "history.txt"), "w") as f:
        f.write(str(history))

    # Save eval result
    with open(os.path.join(model_path, "eval_result.txt"), "w") as f:
        f.write(str(eval_result))

    # Save confusion matrix separately (binary format)
    np.save(os.path.join(model_path, "confusion_matrix.npy"), eval_result["cm"])

In [5]:
def pipeline(model, loaders, optimizer, criterion, n_train_epoch,
             device, save_path, model_name="model", patience=3):

    train_loader, val_loader, test_loader = loaders

    model, history = train(
        model, train_loader, val_loader,
        optimizer, criterion,
        device, n_train_epoch, patience
    )

    eval_result = eval(model, test_loader, device)

    save_result_to_drive(save_path, model_name, model, history, eval_result)

    return model, history, eval_result

In [14]:
num_classes = len(classes)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
max_epochs = 10

In [15]:
model_dict = {}

In [17]:
import torch.nn as nn

In [18]:
from torchvision.models import resnet18, ResNet18_Weights

resnet_model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)

resnet_model.fc = nn.Linear(resnet_model.fc.in_features, num_classes)

# Freeze backbone
for param in resnet_model.parameters():
    param.requires_grad = False

# Unfreeze classifier
for param in resnet_model.fc.parameters():
    param.requires_grad = True

optimizer = torch.optim.Adam(resnet_model.fc.parameters(), lr=1e-3)
model_dict['resnet18_base'] = (resnet_model, optimizer)

In [19]:
model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, num_classes)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
model_dict['resnet18_all'] = (model, optimizer)

In [20]:
model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)

model.fc = nn.Linear(model.fc.in_features, num_classes)

optimizer = torch.optim.Adam([
    {"params": model.layer1.parameters(), "lr": 1e-5},
    {"params": model.layer2.parameters(), "lr": 1e-5},
    {"params": model.layer3.parameters(), "lr": 5e-5},
    {"params": model.layer4.parameters(), "lr": 1e-4},
    {"params": model.fc.parameters(), "lr": 1e-3},
])
model_dict['resnet18_difflr'] = (model, optimizer)

In [21]:
from torchvision.models import vit_b_16, ViT_B_16_Weights
import torch.nn as nn
import torch.optim as optim

model = vit_b_16(weights=ViT_B_16_Weights.IMAGENET1K_V1)

model.heads.head = nn.Linear(model.heads.head.in_features, num_classes)

# freeze backbone
for param in model.parameters():
    param.requires_grad = False

# train classifier
for param in model.heads.parameters():
    param.requires_grad = True

optimizer = optim.AdamW(model.heads.parameters(), lr=1e-3)
model_dict['vit'] = (model, optimizer)

Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth


100%|██████████| 330M/330M [00:02<00:00, 147MB/s]


In [22]:
criterion = torch.nn.CrossEntropyLoss()
loaders = (trainloader, valloader, testloader)

In [ ]:
for model_name, (model, optimizer) in model_dict.items():
    trained_model, history, eval_result = pipeline(
        model=model,
        loaders=loaders,
        optimizer=optimizer,
        criterion=criterion,
        n_train_epoch=max_epochs,
        device=device,
        save_path=ic_path,
        model_name=model_name,
        patience=3
    )